# Multi-Touch Attribution (MTA) Analysis

This notebook demonstrates the full MTA attribution workflow:

1. Generate synthetic marketing data
2. Run all attribution models
3. Compare model results
4. Analyze top converting journeys
5. Visualize channel performance

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.synthetic_generator import SyntheticDataGenerator
from models.attribution import AttributionEngine, MODEL_LABELS, MODELS

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Generate Synthetic Marketing Data

In [ ]:
gen = SyntheticDataGenerator(seed=42)
df = gen.generate(n_users=5000, start_date='2024-01-01', end_date='2024-12-31')

print(f'Total touchpoints: {len(df):,}')
print(f'Unique users: {df["user_id"].nunique():,}')
print(f'Converting journeys: {df[df["converted"]==1]["journey_id"].nunique():,}')
print(f'Total revenue: ${df["revenue"].sum():,.0f}')
df.head(10)

In [ ]:
# Channel distribution
channel_counts = df['channel'].value_counts().reset_index()
channel_counts.columns = ['Channel', 'Touchpoints']

fig = px.bar(channel_counts, x='Channel', y='Touchpoints',
             title='Touchpoint Distribution by Channel',
             color='Channel')
fig.show()

## 2. Run Attribution Models

In [ ]:
engine = AttributionEngine(decay_rate=7, position_first_weight=0.4, position_last_weight=0.4)
results = engine.run_all(df)

# Show results for each model
for model, mdf in results.items():
    print(f'\n=== {MODEL_LABELS[model]} ===')
    print(mdf[['channel', 'attributed_revenue', 'attributed_conversions', 'roas', 'roi']].to_string(index=False))

## 3. Compare Models — Attributed Revenue by Channel

In [ ]:
channels = sorted(set(ch for mdf in results.values() for ch in mdf['channel']))

fig = go.Figure()
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A']
for (model, mdf), color in zip(results.items(), colors):
    values = [float(mdf[mdf['channel']==ch]['attributed_revenue'].iloc[0]) if not mdf[mdf['channel']==ch].empty else 0 for ch in channels]
    fig.add_trace(go.Bar(name=MODEL_LABELS[model], x=channels, y=values, marker_color=color))

fig.update_layout(barmode='group', title='Attributed Revenue by Channel & Model',
                  yaxis_title='Revenue ($)')
fig.show()

## 4. Revenue Attribution Heatmap

In [ ]:
model_names = [MODEL_LABELS[m] for m in results]
matrix = []
for model, mdf in results.items():
    row = [float(mdf[mdf['channel']==ch]['attributed_revenue'].iloc[0]) if not mdf[mdf['channel']==ch].empty else 0 for ch in channels]
    matrix.append(row)

fig = go.Figure(data=go.Heatmap(
    z=matrix, x=channels, y=model_names,
    colorscale='Blues',
    text=[[f'${v:,.0f}' for v in row] for row in matrix],
    texttemplate='%{text}'
))
fig.update_layout(title='Attribution Revenue Heatmap (Models × Channels)')
fig.show()

## 5. Top Converting Journeys

In [ ]:
top_journeys = AttributionEngine.get_top_journeys(df, top_n=15)
print('Top 15 Converting Journey Paths:')
print(top_journeys.to_string(index=False))

In [ ]:
fig = px.bar(top_journeys, x='count', y='journey_path', orientation='h',
             color='avg_revenue', color_continuous_scale='Blues',
             title='Top 15 Converting Journey Paths',
             labels={'count': 'Conversions', 'journey_path': 'Path', 'avg_revenue': 'Avg Revenue ($)'})
fig.update_layout(height=500, yaxis={'autorange': 'reversed'}, margin=dict(l=250))
fig.show()

## 6. Channel ROI & ROAS Comparison

In [ ]:
# Use Linear model for a balanced view
linear_df = results['linear'].sort_values('roas', ascending=False)

fig = make_subplots(rows=1, cols=2, subplot_titles=['ROI by Channel', 'ROAS by Channel'])

fig.add_trace(go.Bar(x=linear_df['channel'], y=linear_df['roi'], name='ROI',
                     marker_color='#4C72B0'), row=1, col=1)
fig.add_trace(go.Bar(x=linear_df['channel'], y=linear_df['roas'], name='ROAS',
                     marker_color='#DD8452'), row=1, col=2)

fig.update_layout(title='Channel ROI & ROAS (Linear Attribution)', showlegend=False)
fig.show()

## 7. Export Results

In [ ]:
import os

frames = []
for model, mdf in results.items():
    frame = mdf.copy()
    frame.insert(0, 'model', MODEL_LABELS[model])
    frames.append(frame)

all_results = pd.concat(frames, ignore_index=True)
output_path = '/tmp/mta_attribution_results.csv'
all_results.to_csv(output_path, index=False)
print(f'Results exported to {output_path}')
all_results.head(10)